# Noisy TE-PAI vs Trotter — time-evolution expectation (6 qubits)

A sanity check **before** shadow spectroscopy: estimate $\langle Z_0(t)\rangle$ of a
6-qubit Heisenberg chain. The two noisy estimates use the **same number of single-shot
measurements** and are shown as the shot mean with a **95% confidence interval** ($1.96\times$
standard error, normal/CLT approximation; valid for large `M`).

**Several noise models are compared side by side**: set `noise_kinds` in the parameter cell
(e.g. `["depolarizing", "amplitude_damping"]`) and each channel gets its own subplot, sharing the
same noise-free reference. Available channels: `depolarizing` (paper Fig. 2, unital),
`amplitude_damping` (T1, non-unital — $\langle Z_0\rangle$ drifts toward $+1$, the $|0\rangle$
state), `phaseflip` (T2 dephasing), `bitflip`. This shows the TE-PAI robustness is not specific to
the depolarizing channel.

Time discretisation: the whole interval $[0, t_{\max}]$ is split into `N` steps of equal width
$\delta t = t_{\max}/N$ (same for every point). Snapshot times land **exactly on the step grid**
(`n_snap` divides `N`, `stride = N / n_snap` steps per snapshot).

- **deep Trotter, noise-free** -- the (near-)exact dynamics, reference line (shared across panels).
- **Trotter noisy** -- one circuit per time point: exact noisy **density matrix**, then `M` shots.
- **TE-PAI noisy** -- `M` shallow circuits, 1 trajectory shot each.

The **cost cell runs first** (overhead $\gamma$ and gate counts, cheap to compute) so you can
size the run before launching the heavy sweep. TE-PAI circuits are much shallower, so TE-PAI
noisy keeps closer to the noise-free curve than Trotter noisy — the Fig. 2 effect.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pai_shadow.hamil import Heisenberg_Hamil
from pai_shadow.circuit import NoiseSpec
from pai_shadow.trotter import trotter_circuit
from pai_shadow.te_pai import TEPAI

n_qubits = 6
H = Heisenberg_Hamil(n_qubits, 1.0, 1.0, 1.0)

# Neel initial state |010101>
neel = [q % 2 for q in range(n_qubits)]
idx = sum(b << q for q, b in enumerate(neel))
init = np.zeros(1 << n_qubits, dtype=complex)
init[idx] = 1.0

observable = "Z" + "I" * (n_qubits - 1)      # <Z_0(t)>
delta = np.pi / 2**5                          # TE-PAI angle
t_max = 3.0
N = 100                                       # divisions of the whole [0, t_max]
n_snap = 20                                   # time snapshots; MUST divide N (-> land on grid)
N_ref = 2000                                  # deep Trotter for the exact-dynamics reference

# --- noise models to compare (each gets its own subplot, plotted side by side) ---
#   "depolarizing"      -- paper Fig 2 (unital; decoheres toward the maximally mixed state)
#   "amplitude_damping" -- T1 relaxation (non-unital; <Z_0> drifts toward +1, the |0> state)
#   "phaseflip"         -- T2 dephasing;   "bitflip" -- X errors
noise_kinds = ["depolarizing", "amplitude_damping"]
p1, p2 = 1e-4, 1e-3                            # 1-qubit / 2-qubit error rates (paper levels)
M = 10000                                     # single-shot budget per time point (noisy methods)
Z95 = 1.959964                                # 95% CI = Z95 * standard error

assert N % n_snap == 0, "n_snap must divide N so snapshots land on step boundaries"
dt = t_max / N                                # constant step width
dt_ref = t_max / N_ref
stride = N // n_snap                          # steps between consecutive snapshots
step_counts = np.arange(0, N + 1, stride)     # 0, stride, ..., N  (integer steps per snapshot)
times = step_counts * dt                      # snapshot times, exactly on the step grid
l1 = float(sum(abs(np.real(c)) for _, _, c in H.get_term(0.0)))   # ||H||_1
print(f"noise_kinds={noise_kinds}, N={N}, dt={dt:.4f}, n_snap={n_snap}, stride={stride} steps,  theta=2*dt={2 * dt:.3f} (<= delta {delta:.3f})")

In [ ]:
# COST PREVIEW (cheap: only samples circuits, no time evolution / measurement).
#   Trot_gates  = n_steps(t) * (#terms)           [noisy Trotter, dt = t_max/N]
#   deep_gates  = n_steps_ref(t) * (#terms)       [reference circuit, dt_ref = t_max/N_ref]
#   gamma(dt)   = TEPAI.overhead;  gamma(lim) = exp(2 ||H||_1 t tan(delta/2))   [dt -> 0]
#   TEPAI_g(dt) = mean sampled gate count;  TEPAI_g(lim) = csc(d)(3-cos d) ||H||_1 t
n_terms = len(H)
print(f"dt = {dt:.4f} (same for all t),  #terms = {n_terms}\n")
print(f"{'t':>5} {'Trot_gates':>11} {'deep_gates':>11} {'gamma(dt)':>10} {'gamma(lim)':>11} {'TEPAI_g(dt)':>12} {'TEPAI_g(lim)':>13}")
for t in [1.0, 2.0, 3.0]:
    ns = round(t / dt)
    ns_ref = round(t / dt_ref)
    tp = TEPAI(H, delta, t, ns, init_state=init)
    circuits, _ = tp.sample(500, rng=np.random.default_rng(0))
    nu = float(np.mean([len(c) for c in circuits]))
    gamma_lim = np.exp(2 * l1 * t * np.tan(delta / 2))
    nu_lim = (3 - np.cos(delta)) / np.sin(delta) * l1 * t
    print(f"{t:5.1f} {ns * n_terms:11d} {ns_ref * n_terms:11d} {tp.overhead:10.3f} {gamma_lim:11.3f} {nu:12.1f} {nu_lim:13.1f}")

In [ ]:
def ci95(samples):
    # mean and 95% confidence half-width (1.96 * standard error)
    samples = np.asarray(samples, dtype=float)
    return samples.mean(), Z95 * samples.std() / np.sqrt(len(samples))

# deep Trotter, noise-free reference (shared by all noise models)
deep_nf = [trotter_circuit(H, t, max(1, round(t / dt_ref)), init_state=init).expectation(observable)
           for t in times]

def run_sweep(noise):
    """One noise channel: Trotter noisy (exact density matrix + M shots) and
    TE-PAI noisy (M trajectory shots) at every snapshot time. Returns the four
    lists (Trotter mean/err, TE-PAI mean/err)."""
    trot, trot_e, tepai, tepai_e = [], [], [], []
    for i, (t, ns) in enumerate(zip(times, step_counts)):
        ns = int(ns)
        circ = trotter_circuit(H, t, max(1, ns), init_state=init)
        dm = circ.evolved_density(noise)             # exact noisy density matrix
        z = [1 - 2 * (v & 1) for v in dm.sampling(M, 1000 + i)]   # qubit 0 = LSB
        m, e = ci95(z); trot.append(m); trot_e.append(e)
        if ns == 0:
            tepai.append(trot[-1]); tepai_e.append(0.0); continue
        tp = TEPAI(H, delta, t, ns, init_state=init)
        m, e = ci95(tp.estimate(observable, M, shots=1, noise=noise, seed=0))   # TE-PAI noisy
        tepai.append(m); tepai_e.append(e)
    return trot, trot_e, tepai, tepai_e

results = {}                                          # noise_kind -> (trot, trot_e, tepai, tepai_e)
for kind in noise_kinds:
    results[kind] = run_sweep(NoiseSpec(p1=p1, p2=p2, kind=kind))
    trot, _, tepai, _ = results[kind]
    print(f"{kind:18s}: max |TE-PAI noisy - Trotter noisy| = "
          f"{np.max(np.abs(np.array(tepai) - np.array(trot))):.3f}")

In [ ]:
fig, axes = plt.subplots(1, len(noise_kinds), figsize=(7.0 * len(noise_kinds), 4.5), sharey=True)
axes = np.atleast_1d(axes)
for ax, kind in zip(axes, noise_kinds):
    trot, trot_e, tepai, tepai_e = results[kind]
    ax.plot(times, deep_nf, color="0.5", ls="--", lw=1.5, label=f"deep Trotter, noise-free (dt={dt_ref:.4f})")
    ax.errorbar(times, trot, yerr=trot_e, color="#1f77b4", ls="none", marker="^", ms=5, capsize=2,
                label=f"Trotter noisy (density, {M} shots)")
    ax.errorbar(times, tepai, yerr=tepai_e, color="#d62728", ls="none", marker="o", ms=4, capsize=2,
                label=f"TE-PAI noisy (trajectory, M={M})")
    ax.set_xlabel("t"); ax.set_title(f"{kind} noise")
    ax.legend(fontsize=8)
axes[0].set_ylabel(r"$\langle Z_0(t)\rangle$")
fig.suptitle(f"{n_qubits}-qubit Heisenberg, Neel init, noisy expectation (same shots, 95% CI)")
fig.tight_layout(); plt.show()